# Step 4: Format and Validate the Final Training Dataset

This notebook:
1. Loads `data/dataset_raw.json`
2. Validates and filters examples
3. Formats into **ChatML** instruction-tuning format
4. Creates 90/10 train/validation split
5. Saves `data/train.jsonl` and `data/val.jsonl`

## ChatML Format (Qwen3)
```
<|im_start|>system
{system_prompt}<|im_end|>
<|im_start|>user
{instruction}\n\n{narrative}<|im_end|>
<|im_start|>assistant
{json_output}<|im_end|>
```

The model learns to produce a structured JSON object from a clinical narrative.

In [ ]:
%pip install pandas tqdm -q

In [ ]:
import json
import random
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm

DATA_DIR      = Path("data")
INPUT_FILE    = DATA_DIR / "dataset_raw.json"
TRAIN_FILE    = DATA_DIR / "train.jsonl"
VAL_FILE      = DATA_DIR / "val.jsonl"

TRAIN_RATIO   = 0.90
RANDOM_SEED   = 42
MIN_NARRATIVE_LEN = 100

SYSTEM_PROMPT = (
    "You are an expert pharmacovigilance medical reviewer. "
    "Analyze clinical safety narratives and provide structured assessments "
    "according to ICH E2D guidelines and MedDRA coding standards."
)

INSTRUCTION = (
    "Analyze the following clinical safety narrative. "
    "Provide a structured medical review covering:\n"
    "1. Seriousness assessment with specific ICH E2D criteria\n"
    "2. MedDRA coding: Preferred Term (PT) and System Organ Class (SOC)\n"
    "3. Labelling status: Expected or Unexpected, with brief evidence\n"
    "4. Causality: WHO-UMC category\n"
    "Return your assessment as a JSON object."
)

print("Config loaded.")

In [ ]:
# ── Load raw data ────────────────────────────────────────────────────────────
with open(INPUT_FILE) as f:
    raw_records = json.load(f)

print(f"Loaded {len(raw_records)} raw records")

In [ ]:
# ── Validation ────────────────────────────────────────────────────────────────

VALID_SERIOUSNESS = {"Serious", "Non-serious"}
VALID_WHO_UMC     = {"Certain", "Probable/Likely", "Possible", "Unlikely",
                     "Conditional/Unclassified", "Unassessable"}
VALID_LABELLING   = {"Expected", "Unexpected"}

def validate_record(rec: dict) -> tuple[bool, str]:
    """Returns (is_valid, reason_if_invalid)."""
    narrative = rec.get("narrative", "")
    if len(narrative) < MIN_NARRATIVE_LEN:
        return False, f"Narrative too short ({len(narrative)} chars)"

    if not rec.get("meddra_pt"):
        return False, "Missing meddra_pt"

    if rec.get("seriousness") not in VALID_SERIOUSNESS:
        return False, f"Invalid seriousness: {rec.get('seriousness')}"

    if rec.get("who_umc_category") not in VALID_WHO_UMC:
        return False, f"Invalid who_umc_category: {rec.get('who_umc_category')}"

    if rec.get("labelling_status") not in VALID_LABELLING:
        return False, f"Invalid labelling_status: {rec.get('labelling_status')}"

    return True, ""


valid_records = []
skip_reasons = {}

for rec in raw_records:
    ok, reason = validate_record(rec)
    if ok:
        valid_records.append(rec)
    else:
        skip_reasons[reason] = skip_reasons.get(reason, 0) + 1

print(f"Valid records: {len(valid_records)} / {len(raw_records)} ({len(valid_records)/len(raw_records):.1%})")
if skip_reasons:
    print("\nSkip reasons:")
    for reason, count in sorted(skip_reasons.items(), key=lambda x: -x[1]):
        print(f"  {count:4d} × {reason}")

In [ ]:
# ── Format into ChatML ────────────────────────────────────────────────────────

def format_output(rec: dict) -> dict:
    """Build the JSON output object the model should produce.
    MedDRA codes are included here (model learns to output them).
    WHO-UMC comes from FAERS — deterministic ground truth.
    Naranjo is dropped.
    """
    return {
        "seriousness":          rec["seriousness"],
        "seriousness_criteria": rec.get("seriousness_criteria", []),
        "meddra_pt":            rec["meddra_pt"],
        "meddra_pt_code":       rec.get("meddra_pt_code"),
        "meddra_soc":           rec.get("meddra_soc", ""),
        "meddra_soc_code":      rec.get("meddra_soc_code"),
        "labelling_status":     rec["labelling_status"],
        "labelling_evidence":   rec.get("labelling_evidence", ""),
        "who_umc_category":     rec["who_umc_category"],
    }


def to_chatml(rec: dict) -> dict:
    """Format a record as a ChatML conversation for SFT."""
    output_json = json.dumps(format_output(rec), indent=2)
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"{INSTRUCTION}\n\nNarrative:\n{rec['narrative']}"},
            {"role": "assistant", "content": output_json},
        ]
    }


def to_flat_text(rec: dict) -> dict:
    """Format as a single text string in ChatML template (for SFTTrainer)."""
    output_json = json.dumps(format_output(rec), indent=2)
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{INSTRUCTION}\n\nNarrative:\n{rec['narrative']}<|im_end|>\n"
        f"<|im_start|>assistant\n{output_json}<|im_end|>"
    )
    return {"text": text}


# Test
sample = to_chatml(valid_records[0])
print("Sample ChatML format:")
print(f"  system: {sample['messages'][0]['content'][:80]}...")
print(f"  user:   {sample['messages'][1]['content'][:120]}...")
print(f"  asst:   {sample['messages'][2]['content'][:300]}...")

In [ ]:
# ── Token length check ────────────────────────────────────────────────────────
# Estimate token count (rough: 1 token ≈ 4 chars)
flat_samples = [to_flat_text(r) for r in valid_records]
lengths = [len(s["text"]) // 4 for s in flat_samples]

df_len = pd.Series(lengths)
print("Estimated token length distribution:")
print(df_len.describe().round(0).to_string())
print(f"\nExamples > 2048 tokens: {(df_len > 2048).sum()} ({(df_len > 2048).mean():.1%})")
print(f"Examples > 1024 tokens: {(df_len > 1024).sum()} ({(df_len > 1024).mean():.1%})")
print("\n(Training uses max_seq_length=2048 — examples over this will be truncated)")

In [ ]:
# ── Train / validation split ──────────────────────────────────────────────────
random.seed(RANDOM_SEED)
shuffled = list(valid_records)
random.shuffle(shuffled)

n_train = int(len(shuffled) * TRAIN_RATIO)
train_recs = shuffled[:n_train]
val_recs   = shuffled[n_train:]

print(f"Train: {len(train_recs)} examples")
print(f"Val:   {len(val_recs)} examples")

# Check class balance in splits
for split_name, split in [("train", train_recs), ("val", val_recs)]:
    serious_pct = sum(1 for r in split if r["seriousness"] == "Serious") / len(split)
    expected_pct = sum(1 for r in split if r["labelling_status"] == "Expected") / len(split)
    print(f"  {split_name}: {serious_pct:.1%} serious | {expected_pct:.1%} expected labelling")

In [ ]:
# ── Save JSONL files ──────────────────────────────────────────────────────────
def save_jsonl(records: list, path: Path, formatter):
    with open(path, "w") as f:
        for rec in records:
            f.write(json.dumps(formatter(rec)) + "\n")
    print(f"Saved {len(records)} examples to {path} ({path.stat().st_size / 1e3:.0f} KB)")

# Save both formats: flat text (for SFTTrainer) and ChatML messages (for evaluation)
save_jsonl(train_recs, TRAIN_FILE, to_flat_text)
save_jsonl(val_recs,   VAL_FILE,   to_flat_text)

# Also save ChatML format for evaluation scripts
save_jsonl(train_recs, DATA_DIR / "train_chatml.jsonl", to_chatml)
save_jsonl(val_recs,   DATA_DIR / "val_chatml.jsonl",   to_chatml)

In [ ]:
# ── Final statistics ─────────────────────────────────────────────────────────
df = pd.DataFrame(valid_records)

print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)
print(f"  Total examples:       {len(df)}")
print(f"  Train / Val split:    {len(train_recs)} / {len(val_recs)}")
print()
print(f"  Seriousness:")
print(df["seriousness"].value_counts().to_string(header=False))
print(f"\n  Labelling status:")
print(df["labelling_status"].value_counts().to_string(header=False))
print(f"\n  Labelling source (fda_label vs llm_knowledge):")
if "label_source" in df.columns:
    print(df["label_source"].value_counts().to_string(header=False))
print(f"\n  WHO-UMC categories:")
print(df["who_umc_category"].value_counts().to_string(header=False))
print(f"\n  Top 5 MedDRA SOCs:")
print(df["meddra_soc"].value_counts().head(5).to_string(header=False))
print(f"\n  Narrative sources:")
print(df["narrative_source"].value_counts().to_string(header=False))
print()
print("Output files:")
for f_path in [TRAIN_FILE, VAL_FILE, DATA_DIR / "train_chatml.jsonl", DATA_DIR / "val_chatml.jsonl"]:
    print(f"  {f_path}")

## Done

Training data is ready:
- `data/train.jsonl` — flat text format for `SFTTrainer`
- `data/val.jsonl` — validation set

Proceed to fine-tuning:
```bash
cd train
python train.py
```